In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns                                                                                                                                                                                                                                                       

In [ ]:
import os
import re

In [ ]:
experiment_series ="systems_pop8"
precipitation_setting = "Nieselregen_demonstrator"
decay_setting = "decay"

data_path = f"/media/iru-mls/My Book/pop8_results_2026/raw/{experiment_series}/{decay_setting}_{precipitation_setting}"

experiment_series = "pop8"
result_path = f"preprocessed_data/{experiment_series}/"

## substances

In [ ]:
def tidy_substances(data_path, memilio_id):
    # Path to text file
    file_name = f'INSIDe_substances_results{memilio_id}_output_v4.txt'
    file_path = f"{data_path}/{file_name}"

    # Initialize containers
    records = []
    current_variable = None
    current_manhole = None

    # Regular expressions
    # manhole IDs now come as MW### or RW###, with three digits.
    pattern_str = fr'INSIDe_substances_results_{memilio_id}_output_v4_manhole_((?:MW|RW)\d{{3}})\.txt'
    manhole_pattern = re.compile(pattern_str)
    header_pattern = re.compile(r'time\[min\]\s+(\w+)\([^)]+\) concentration')
    data_pattern = re.compile(r'^(\d+)\s+([-\d.eE]+)$')

    # Read file line by line
    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()

            # Skip empty lines
            if not line or line == '##':
                continue

            # Match manhole ID
            manhole_match = manhole_pattern.match(line)
            if manhole_match:
                current_manhole = manhole_match.group(1)
                continue

            # Match variable name
            header_match = header_pattern.match(line)
            if header_match:
                current_variable = header_match.group(1)
                continue

            # Match data lines
            data_match = data_pattern.match(line)
            if data_match and current_variable and current_manhole:
                time = int(data_match.group(1))
                value = float(data_match.group(2))
                records.append({
                    "time_in_minutes": time,
                    "variable": current_variable,
                    "value": value,
                    "manhole": current_manhole
                })
    df = pd.DataFrame(records)
    df = df.loc[df.variable!="T"]
    df["time_in_days"] = df["time_in_minutes"]/(24*60)
    df["simulation_id"] = memilio_id
    return df

In [ ]:
df = pd.DataFrame()
# Memilio IDs now range from 0 to 100 inclusive
for memilio_id in range(0, 101):
    print(f"Processing Memilio ID: {memilio_id}")
    df_temp = tidy_substances(data_path, memilio_id)
    df = pd.concat([df, df_temp], ignore_index=True)

df = df.loc[df.manhole.isin(["MW022", "MW023", "MW017", "MW043", "MW048",
"RW157", "MW046", "MW061", "RW143", "RW141",
"RW155", "MW059", "RW211", "MW054",
"RW126", "MW052"])]

df.manhole = df.manhole.map({"MW022": "1", "MW023": "2", "MW017": "3", "MW043": "4", "MW048": "5",
"RW157": "6", "MW046": "7", "MW061": "8", "RW143": "9", "RW141": "10",
"RW155": "11", "MW059": "12", "RW211": "13", "MW054": "14",
"RW126": "15", "MW052": "16"})

os.makedirs(f"{result_path}/substances/", exist_ok=True)
df.to_csv(f"{result_path}/substances/{decay_setting}_{precipitation_setting}_output.csv", index=False)

In [ ]:
df.manhole.nunique()

In [ ]:
df.groupby("manhole")["time_in_minutes"].count().describe()["std"]==0.0

In [ ]:
df.loc[df.variable=="PMMoV", "value"].describe()

In [ ]:
df.loc[df.variable=="PMMoV"].groupby("simulation_id")["value"].mean().min()